In [17]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel, AutoTokenizer, AutoModelForCausalLM
import numpy as np
import torch

In [18]:
model = GPT2LMHeadModel.from_pretrained("gpt2")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model.eval()

C:\Users\Makai\AppData\Roaming\Python\Python312\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2SdpaAttention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [5]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B")
model.eval()

Loading checkpoint shards: 100%|██████████| 4/4 [00:19<00:00,  4.79s/it]


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (n

In [6]:
with open('./in.txt', 'r') as file:
    data = [line.strip() for line in file]

In [7]:
def calculate_perplexity(text):
    inputs = tokenizer.encode(text, return_tensors="pt")
    with torch.no_grad():
        outputs = model(inputs, labels=inputs)
        loss = outputs.loss
        perplexity = torch.exp(loss)
    return perplexity.item()

In [19]:
def calculate_log_likelihood_ratio(text):
    inputs = tokenizer(text, return_tensors="pt")
    input_ids = inputs["input_ids"]

    # Calculate log-likelihood for each token chunk
    with torch.no_grad(): 
        log_likelihoods = []
        for i in range(1, len(input_ids[0]) - 1):
            partial_input = input_ids[:, :i + 1]
            outputs = model(partial_input, labels=partial_input)
            log_likelihoods.append(-outputs.loss.item())
        avg_likelihood = sum(log_likelihoods) / len(log_likelihoods)
    return avg_likelihood

likelihood_threshold = -0.3

In [12]:
def get_coherence(text, window_size=4096):
    tokens = tokenizer("\n" + text + "\n", return_tensors='pt', truncation=True, max_length=window_size)
    input_ids = tokens.input_ids
    
    short_context = input_ids[:, :window_size // 2]
    
    with torch.no_grad():
        outputs_short = model(short_context, labels=short_context)
    
    coherence_acc_short = (outputs_short.logits.argmax(-1) == short_context).float().mean().item()
    
    return coherence_acc_short


In [ ]:
calculate_log_likelihood_ratio(data[6])

-3.9046078750065396

In [28]:
perplexity_data = []
for text in data:
    try:
        perplexity = calculate_perplexity(text)
    except Exception as e:
        perplexity = 0
    perplexity_data.append(perplexity)

for data_point in perplexity_data:
    print(data_point)

58.48974609375
15.513893127441406
15.915287017822266
12.15900707244873
18.939476013183594
16.250871658325195
21.423004150390625
19.491636276245117
24.93813133239746
20.15383529663086
21.962215423583984
23.825349807739258
25.870065689086914
25.858251571655273
22.301029205322266
32.906211853027344
21.05856704711914
32.5257568359375
23.495891571044922
22.34467124938965
14.117883682250977
25.817028045654297
27.735980987548828
14.420408248901367
15.449254035949707
24.524974822998047
28.162384033203125
23.552082061767578
25.91586685180664
26.401996612548828
30.054855346679688
27.64107894897461
28.748807907104492
29.741640090942383
25.605859756469727
22.495532989501953
13.887131690979004
20.866884231567383
18.306730270385742
15.58747673034668
27.41030502319336
26.764406204223633
28.813302993774414
29.989774703979492
30.781173706054688
34.46858596801758
32.5933952331543
30.813928604125977
32.18511199951172
29.411943435668945
30.072208404541016
29.370319366455078
35.3016357421875
22.35117721557

In [29]:
ratio_data = []
for text in data:
    try:
        ratio = calculate_log_likelihood_ratio(text)
    except Exception as e:
        ratio = 0
    ratio_data.append(ratio)
    
for data_point in ratio_data:
    print(data_point)

-3.7336743474006653
-3.868847029549735
-3.868847029549735
-3.868847029549735
-3.868847029549735
-3.9046078750065396
-3.9046078750065396
-3.9046078750065396
-3.9046078750065396
-3.9046078750065396
-3.9403118746621266
-3.9403118746621266
-3.9403118746621266
-3.9403118746621266
-3.9403118746621266
-3.920966863632202
-3.920966863632202
-3.920966863632202
-3.920966863632202
-3.920966863632202
-3.8929416111537387
-3.8929416111537387
-3.8929416111537387
-3.8929416111537387
-3.8929416111537387
-3.92949492590768
-3.92949492590768
-3.92949492590768
-3.92949492590768
-3.92949492590768
-3.9462599754333496
-3.9462599754333496
-3.9462599754333496
-3.9462599754333496
-3.9462599754333496
-3.946406466620309
-3.946406466620309
-3.946406466620309
-3.946406466620309
-3.946406466620309
-4.130729238192241
-4.130729238192241
-4.130729238192241
-4.130729238192241
-4.130729238192241
-3.964519432612828
-3.964519432612828
-3.964519432612828
-3.964519432612828
-3.964519432612828
-4.1298482077462335
-4.12984820774

In [36]:
all_data = [(d, p, r) for d, p, r in zip(data, perplexity_data, ratio_data)]

In [58]:
sorted_all_data = sorted(all_data, key=lambda x: x[1], reverse=True)

for item in sorted_all_data:
    print(f"{item[0]}\t\t {item[1]}\t\t {item[2]}")

Once upon a time there usedÂ be two		 968.06103515625		 -5.497474551200867
Once upon a time there usedÂ  be this		 886.4288940429688		 -5.516610145568848
Once upon a time there usedÂ  be		 884.4124755859375		 -5.335422515869141
Once upon a time there usedÂ be this		 817.9384765625		 -5.497474551200867
Once upon a time there used the exist two		 680.5313720703125		 -4.983962740216937
Once upon a time there usedÂ a be		 673.08544921875		 -5.458948969841003
Once upon a time there usedÂ beÂ		 655.8773803710938		 -5.497474551200867
Once upon a time there usedÂ be an		 643.7085571289062		 -5.497474551200867
Once upon a time there usedÂ  be an		 616.4830322265625		 -5.516610145568848
Once upon a time there usedÂ  beÂ		 598.7036743164062		 -5.516610145568848
Once upon a time there used the exist an		 560.8195190429688		 -4.983962740216937
Once upon a time there usedÂ a place		 559.130615234375		 -5.458948969841003
Once upon a time there usedÂ aÂ		 529.6708984375		 -5.458948969841003
Once upon 

In [64]:
trimmed_all_data = [item for item in sorted_all_data if 30 <= item[1] <= 60]

for item in trimmed_all_data:
    print(f"{item[0]}\t\t {item[1]}\t\t {item[2]}")

Once upon a time I used the following function		 59.97846221923828		 -4.391883168901716
Once upon a time in the faraway country		 59.93106460571289		 -4.363173314503261
Once upon a time there lived two brothers named		 59.6123046875		 -4.554061344691685
Once upon a time in a land not so		 59.551788330078125		 -4.478455679757254
Once upon a time, in 1987		 59.524505615234375		 -4.435415649414063
Once upon a time in an ancient world far		 59.44905471801758		 -4.46052748816354
Once upon a time there existed the idea for		 59.43193435668945		 -4.448128529957363
Once upon a time in a land that wasn		 59.369140625		 -4.421884945460728
Once upon a time I used a 2		 59.311614990234375		 -4.49754802385966
Once upon a time in the land far,		 59.305816650390625		 -4.358054127011981
Once upon a time there was the first computer		 59.2923583984375		 -4.194271768842425
Once upon a time there were three bears:		 59.221492767333984		 -4.296806641987392
Once upon a time in a galaxy very near		 59.15093